# Real-world Application and Business Value

## Introduction

In previous notebooks, we've built and deployed machine learning models to predict house prices in Ames, Iowa. However, the true value of a machine learning model lies in its ability to solve real-world problems and create business value.

This notebook focuses on translating our technical work into business impact by exploring:

1. Cost-benefit analysis of model implementation
2. Risk assessment and uncertainty quantification
3. Decision support systems for real estate stakeholders
4. Case studies applying our model to specific business scenarios

By the end of this notebook, you'll understand how to communicate the value of machine learning models to non-technical stakeholders and make data-driven business decisions in the real estate domain.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import os
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

In [ ]:
# Load our preprocessed data and trained model
try:
    # Try to use IPython magic if in notebook environment
    get_ipython().run_line_magic('run', 'src/preprocessing.py')
except (NameError, AttributeError):
    # If not in notebook environment, import directly
    import sys
    import os
    sys.path.append(os.path.join(os.path.dirname(__file__), 'src'))
    from preprocessing import (
        dataset_2, target_2,
        housing_df  # Original dataframe before preprocessing
    )

In [ ]:
# Load the trained model
models_dir = 'models'
try:
    # Load the optimized pipeline
    optimized_pipeline = joblib.load(os.path.join(models_dir, 'optimized_pipeline.joblib'))
    
    # Load the important features
    with open(os.path.join(models_dir, 'important_features.pkl'), 'rb') as f:
        important_features = pickle.load(f)
    
    print(f"Loaded model with {len(important_features)} important features")
except (FileNotFoundError, OSError):
    print("Model files not found. Please run the previous notebook to create them.")
    # Create a simple model for demonstration purposes
    from sklearn.linear_model import Lasso
    
    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(dataset_2, target_2, test_size=0.2, random_state=42)
    
    # Train a simple model
    optimized_pipeline = Lasso(alpha=100, max_iter=10000)
    optimized_pipeline.fit(X_train, y_train)
    
    # Use all features
    important_features = dataset_2.columns.tolist()
    
    print(f"Created a simple model with {len(important_features)} features")

## 1. Cost-Benefit Analysis

Let's analyze the costs and benefits of implementing our house price prediction model in various business contexts.

### 1.1 Implementation Costs

First, let's estimate the costs associated with developing and deploying our model:

In [ ]:
# Define implementation costs
implementation_costs = {
    'Data collection and cleaning': 5000,  # Cost in USD
    'Model development': 10000,
    'Model validation and testing': 3000,
    'Deployment infrastructure': 2000,
    'Web interface development': 4000,
    'Documentation and training': 3000,
    'Ongoing maintenance (annual)': 5000
}

# Calculate total initial cost and annual cost
initial_cost = sum([cost for item, cost in implementation_costs.items() if 'annual' not in item])
annual_cost = implementation_costs['Ongoing maintenance (annual)']

# Display costs
print(f"Initial implementation cost: ${initial_cost:,}")
print(f"Annual maintenance cost: ${annual_cost:,}")

# Create a bar chart of costs
plt.figure(figsize=(12, 6))
items = [item for item in implementation_costs.keys()]
costs = [cost for cost in implementation_costs.values()]

# Sort by cost (descending)
sorted_indices = np.argsort(costs)[::-1]
sorted_items = [items[i] for i in sorted_indices]
sorted_costs = [costs[i] for i in sorted_indices]

# Plot
bars = plt.bar(sorted_items, sorted_costs, color='skyblue')
plt.xticks(rotation=45, ha='right')
plt.title('Implementation Costs Breakdown', fontsize=16)
plt.ylabel('Cost (USD)', fontsize=14)

# Add cost labels on top of bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 100,
             f'${height:,}',
             ha='center', va='bottom', fontsize=12)

plt.tight_layout()
plt.show()

### 1.2 Potential Benefits

Now, let's estimate the potential benefits for different stakeholders in the real estate market:

In [ ]:
# Calculate model performance metrics on test data
X_train, X_test, y_train, y_test = train_test_split(dataset_2, target_2, test_size=0.2, random_state=42)

# If we have important features, use only those
if len(important_features) < len(dataset_2.columns):
    X_test_selected = X_test[important_features]
else:
    X_test_selected = X_test

# Make predictions
y_pred = optimized_pipeline.predict(X_test_selected)

# Calculate metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100  # Mean Absolute Percentage Error

print(f"Model Performance Metrics:")
print(f"RMSE: ${rmse:.2f}")
print(f"MAE: ${mae:.2f}")
print(f"R² Score: {r2:.4f}")
print(f"MAPE: {mape:.2f}%")

In [ ]:
# Calculate average house price in the dataset
avg_house_price = target_2.mean()
print(f"Average house price in dataset: ${avg_house_price:.2f}")

# Calculate percentage error relative to average house price
mae_percentage = (mae / avg_house_price) * 100
print(f"MAE as percentage of average house price: {mae_percentage:.2f}%")

In [ ]:
# Define potential benefits for different stakeholders
# These are estimated annual benefits based on model performance

# Assumptions
avg_commission_rate = 0.06  # 6% typical real estate commission
avg_transactions_per_agent = 12  # Average number of transactions per agent per year
improved_pricing_benefit = 0.02  # 2% improvement in pricing accuracy
time_savings_hours = 2  # Hours saved per transaction in market analysis
hourly_rate = 50  # Hourly rate for real estate professionals

# Calculate benefits
benefits = {
    'Real Estate Agents': {
        'description': 'More accurate pricing leads to faster sales and higher commissions',
        'calculation': f"${avg_house_price:.0f} × {avg_commission_rate:.1%} × {improved_pricing_benefit:.1%} × {avg_transactions_per_agent} transactions",
        'annual_value': avg_house_price * avg_commission_rate * improved_pricing_benefit * avg_transactions_per_agent,
        'time_savings': f"{time_savings_hours} hours × ${hourly_rate} × {avg_transactions_per_agent} transactions",
        'time_value': time_savings_hours * hourly_rate * avg_transactions_per_agent
    },
    'Home Sellers': {
        'description': 'Optimal pricing leads to faster sales and potentially higher sale prices',
        'calculation': f"${avg_house_price:.0f} × {improved_pricing_benefit:.1%}",
        'annual_value': avg_house_price * improved_pricing_benefit,
        'time_savings': 'Reduced time on market by 15 days',
        'time_value': 0  # Hard to quantify directly
    },
    'Home Buyers': {
        'description': 'Better understanding of fair market value prevents overpaying',
        'calculation': f"${avg_house_price:.0f} × {improved_pricing_benefit:.1%}",
        'annual_value': avg_house_price * improved_pricing_benefit,
        'time_savings': 'Faster decision-making process',
        'time_value': 0  # Hard to quantify directly
    },
    'Appraisers': {
        'description': 'Supplementary tool to validate appraisals',
        'calculation': f"{time_savings_hours} hours × ${hourly_rate} × 50 appraisals",
        'annual_value': 0,  # Primarily time savings
        'time_savings': f"{time_savings_hours} hours × 50 appraisals",
        'time_value': time_savings_hours * hourly_rate * 50
    },
    'Banks/Lenders': {
        'description': 'Reduced risk in mortgage lending through better valuation',
        'calculation': f"${avg_house_price:.0f} × 0.5% × 100 loans",  # 0.5% reduction in default risk
        'annual_value': avg_house_price * 0.005 * 100,
        'time_savings': 'Faster loan processing',
        'time_value': 0  # Hard to quantify directly
    }
}

# Calculate total monetary benefit
total_monetary_benefit = sum([stakeholder['annual_value'] for stakeholder in benefits.values()])
total_time_value = sum([stakeholder['time_value'] for stakeholder in benefits.values()])
total_benefit = total_monetary_benefit + total_time_value

print(f"Total estimated annual benefit: ${total_benefit:,.2f}")
print(f"  - Direct monetary benefit: ${total_monetary_benefit:,.2f}")
print(f"  - Time savings value: ${total_time_value:,.2f}")

In [ ]:
# Create a table of benefits
benefit_table = []
for stakeholder, details in benefits.items():
    benefit_table.append({
        'Stakeholder': stakeholder,
        'Description': details['description'],
        'Monetary Benefit': f"${details['annual_value']:,.2f}",
        'Time Savings': details['time_savings'],
        'Time Value': f"${details['time_value']:,.2f}",
        'Total Value': f"${details['annual_value'] + details['time_value']:,.2f}"
    })

# Convert to DataFrame and display
benefit_df = pd.DataFrame(benefit_table)
benefit_df

In [ ]:
# Create a bar chart of benefits by stakeholder
plt.figure(figsize=(12, 6))

stakeholders = list(benefits.keys())
monetary_benefits = [details['annual_value'] for details in benefits.values()]
time_values = [details['time_value'] for details in benefits.values()]

# Sort by total benefit
total_benefits = [m + t for m, t in zip(monetary_benefits, time_values)]
sorted_indices = np.argsort(total_benefits)[::-1]
stakeholders = [stakeholders[i] for i in sorted_indices]
monetary_benefits = [monetary_benefits[i] for i in sorted_indices]
time_values = [time_values[i] for i in sorted_indices]

# Create stacked bar chart
bar_width = 0.6
plt.bar(stakeholders, monetary_benefits, bar_width, label='Direct Monetary Benefit', color='skyblue')
plt.bar(stakeholders, time_values, bar_width, bottom=monetary_benefits, label='Time Savings Value', color='lightgreen')

plt.title('Estimated Annual Benefits by Stakeholder', fontsize=16)
plt.ylabel('Annual Benefit (USD)', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.legend()

# Add value labels
for i, (m, t) in enumerate(zip(monetary_benefits, time_values)):
    total = m + t
    plt.text(i, total + 100, f'${total:,.0f}', ha='center', va='bottom', fontsize=12)

plt.tight_layout()
plt.show()

### 1.3 Return on Investment (ROI) Analysis

Let's calculate the ROI for our model implementation over a 5-year period:

In [ ]:
# Define ROI calculation parameters
time_horizon = 5  # years
discount_rate = 0.05  # 5% annual discount rate for NPV calculation

# Calculate costs and benefits over time
costs_over_time = [initial_cost] + [annual_cost] * (time_horizon - 1)
benefits_over_time = [total_benefit] * time_horizon

# Calculate net cash flow
net_cash_flow = [benefits_over_time[i] - costs_over_time[i] for i in range(time_horizon)]

# Calculate cumulative cash flow
cumulative_cash_flow = []
running_total = 0
for flow in net_cash_flow:
    running_total += flow
    cumulative_cash_flow.append(running_total)

# Calculate NPV (Net Present Value)
npv = sum([net_cash_flow[i] / (1 + discount_rate) ** i for i in range(time_horizon)])

# Calculate ROI
total_cost = initial_cost + annual_cost * (time_horizon - 1)
total_benefit_5yr = total_benefit * time_horizon
roi = (total_benefit_5yr - total_cost) / total_cost * 100

# Calculate payback period
payback_period = 0
for i, cum_flow in enumerate(cumulative_cash_flow):
    if cum_flow >= 0:
        if i > 0 and cumulative_cash_flow[i-1] < 0:
            # Interpolate for more accurate payback period
            prev_flow = cumulative_cash_flow[i-1]
            payback_period = i - 1 + abs(prev_flow) / (cum_flow - prev_flow)
        else:
            payback_period = i
        break

# Display results
print(f"5-Year ROI Analysis:")
print(f"Total 5-year cost: ${total_cost:,.2f}")
print(f"Total 5-year benefit: ${total_benefit_5yr:,.2f}")
print(f"Net Present Value (NPV): ${npv:,.2f}")
print(f"Return on Investment (ROI): {roi:.2f}%")
print(f"Payback Period: {payback_period:.2f} years")

In [ ]:
# Create a cash flow chart
plt.figure(figsize=(12, 6))

years = list(range(1, time_horizon + 1))

# Plot costs as negative values
plt.bar(years, [-cost for cost in costs_over_time], color='salmon', label='Costs')

# Plot benefits
plt.bar(years, benefits_over_time, color='skyblue', label='Benefits')

# Plot cumulative cash flow
plt.plot(years, cumulative_cash_flow, 'ko-', linewidth=2, label='Cumulative Cash Flow')

# Add zero line
plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)

# Add payback period line
if payback_period <= time_horizon:
    plt.axvline(x=payback_period, color='green', linestyle='--', label=f'Payback Period: {payback_period:.2f} years')

plt.title('5-Year Cash Flow Analysis', fontsize=16)
plt.xlabel('Year', fontsize=14)
plt.ylabel('Cash Flow (USD)', fontsize=14)
plt.xticks(years)
plt.legend()
plt.grid(True, alpha=0.3)

# Add value labels
for i, (cost, benefit, cum_flow) in enumerate(zip(costs_over_time, benefits_over_time, cumulative_cash_flow)):
    year = i + 1
    # Cost label
    plt.text(year, -cost/2, f'-${cost:,.0f}', ha='center', va='center', color='white', fontweight='bold')
    # Benefit label
    plt.text(year, benefit/2, f'${benefit:,.0f}', ha='center', va='center', color='white', fontweight='bold')
    # Cumulative flow label
    plt.text(year, cum_flow + 1000, f'${cum_flow:,.0f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 2. Risk Assessment and Uncertainty Quantification

Let's analyze the risks associated with our model and quantify the uncertainty in its predictions.

### 2.1 Prediction Error Analysis

First, let's analyze the distribution of prediction errors to understand the model's limitations:

In [ ]:
# Calculate prediction errors
errors = y_test - y_pred
percentage_errors = (errors / y_test) * 100

# Create a DataFrame with actual values, predictions, and errors
error_df = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred,
    'Error': errors,
    'Percentage Error': percentage_errors
})

# Display summary statistics
print("Error Statistics:")
print(error_df[['Error', 'Percentage Error']].describe())

In [ ]:
# Create a histogram of errors
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.hist(errors, bins=30, color='skyblue', edgecolor='black', alpha=0.7)
plt.axvline(x=0, color='red', linestyle='--', linewidth=2)
plt.title('Distribution of Prediction Errors', fontsize=14)
plt.xlabel('Error (Actual - Predicted) in $', fontsize=12)
plt.ylabel('Frequency', fontsize=12)

plt.subplot(1, 2, 2)
plt.hist(percentage_errors, bins=30, color='lightgreen', edgecolor='black', alpha=0.7)
plt.axvline(x=0, color='red', linestyle='--', linewidth=2)
plt.title('Distribution of Percentage Errors', fontsize=14)
plt.xlabel('Percentage Error (%)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Create a scatter plot of actual vs. predicted values
plt.figure(figsize=(10, 8))

# Plot the scatter points
plt.scatter(y_test, y_pred, alpha=0.5, color='blue')

# Add perfect prediction line
max_value = max(y_test.max(), y_pred.max())
min_value = min(y_test.min(), y_pred.min())
plt.plot([min_value, max_value], [min_value, max_value], 'r--', linewidth=2)

# Add +/- 10% error bands
plt.plot([min_value, max_value], [min_value*0.9, max_value*0.9], 'g--', linewidth=1, alpha=0.5)
plt.plot([min_value, max_value], [min_value*1.1, max_value*1.1], 'g--', linewidth=1, alpha=0.5)

plt.title('Actual vs. Predicted House Prices', fontsize=16)
plt.xlabel('Actual Price ($)', fontsize=14)
plt.ylabel('Predicted Price ($)', fontsize=14)
plt.grid(True, alpha=0.3)

# Add text annotations
plt.text(min_value, max_value*0.95, f'R² = {r2:.4f}', fontsize=12, bbox=dict(facecolor='white', alpha=0.8))
plt.text(min_value, max_value*0.9, f'MAPE = {mape:.2f}%', fontsize=12, bbox=dict(facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

### 2.2 Risk Categories and Mitigation Strategies

Let's identify and analyze different types of risks associated with our model:

In [ ]:
# Define risk categories and mitigation strategies
risks = [
    {
        'category': 'Data Quality Risk',
        'description': 'Missing or inaccurate data in new properties',
        'probability': 'High',
        'impact': 'Medium',
        'mitigation': 'Implement data validation checks and handle missing values gracefully'
    },
    {
        'category': 'Model Drift Risk',
        'description': 'Model performance degrades over time as market conditions change',
        'probability': 'High',
        'impact': 'High',
        'mitigation': 'Regularly retrain the model with new data and monitor performance metrics'
    },
    {
        'category': 'Outlier Risk',
        'description': 'Unusual properties lead to large prediction errors',
        'probability': 'Medium',
        'impact': 'High',
        'mitigation': 'Implement outlier detection and flag predictions with high uncertainty'
    },
    {
        'category': 'Feature Availability Risk',
        'description': 'Important features may not be available for all properties',
        'probability': 'Medium',
        'impact': 'Medium',
        'mitigation': 'Develop fallback models that use different feature subsets'
    },
    {
        'category': 'Overreliance Risk',
        'description': 'Users rely too heavily on model predictions without expert judgment',
        'probability': 'Medium',
        'impact': 'High',
        'mitigation': 'Clearly communicate model limitations and provide confidence intervals'
    },
    {
        'category': 'Market Shock Risk',
        'description': 'Sudden economic changes invalidate historical patterns',
        'probability': 'Low',
        'impact': 'Very High',
        'mitigation': 'Implement circuit breakers that detect unusual market conditions'
    },
    {
        'category': 'Regulatory Risk',
        'description': 'Changes in regulations affect model validity or usage',
        'probability': 'Low',
        'impact': 'High',
        'mitigation': 'Monitor regulatory developments and ensure model transparency'
    }
]

# Convert to DataFrame and display
risk_df = pd.DataFrame(risks)
risk_df

In [ ]:
# Create a risk matrix
# Convert probability and impact to numeric values
probability_map = {'Low': 1, 'Medium': 2, 'High': 3}
impact_map = {'Low': 1, 'Medium': 2, 'High': 3, 'Very High': 4}

risk_df['probability_value'] = risk_df['probability'].map(probability_map)
risk_df['impact_value'] = risk_df['impact'].map(impact_map)
risk_df['risk_score'] = risk_df['probability_value'] * risk_df['impact_value']

# Create the risk matrix
plt.figure(figsize=(10, 8))

# Define the grid
impact_labels = ['Low', 'Medium', 'High', 'Very High']
probability_labels = ['Low', 'Medium', 'High']

# Create a background color grid
for i in range(1, 5):  # Impact
    for j in range(1, 4):  # Probability
        risk_score = i * j
        if risk_score <= 3:  # Low risk
            color = 'lightgreen'
        elif risk_score <= 6:  # Medium risk
            color = 'khaki'
        elif risk_score <= 9:  # High risk
            color = 'salmon'
        else:  # Very high risk
            color = 'firebrick'
        plt.fill_between([i-0.5, i+0.5], [j-0.5, j-0.5], [j+0.5, j+0.5], color=color, alpha=0.3)

# Plot each risk as a point
for _, risk in risk_df.iterrows():
    plt.scatter(risk['impact_value'], risk['probability_value'], s=200, color='blue', alpha=0.7)
    plt.text(risk['impact_value'], risk['probability_value'], risk['category'].split(' ')[0],
             ha='center', va='center', fontsize=10, fontweight='bold')

# Set up the plot
plt.xlim(0.5, 4.5)
plt.ylim(0.5, 3.5)
plt.xticks(range(1, 5), impact_labels)
plt.yticks(range(1, 4), probability_labels)
plt.xlabel('Impact', fontsize=14)
plt.ylabel('Probability', fontsize=14)
plt.title('Risk Matrix', fontsize=16)
plt.grid(True, alpha=0.3)

# Add a legend for risk levels
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='lightgreen', alpha=0.3, label='Low Risk'),
    Patch(facecolor='khaki', alpha=0.3, label='Medium Risk'),
    Patch(facecolor='salmon', alpha=0.3, label='High Risk'),
    Patch(facecolor='firebrick', alpha=0.3, label='Very High Risk')
]
plt.legend(handles=legend_elements, loc='upper left')

plt.tight_layout()
plt.show()

### 2.3 Confidence Intervals for Predictions

Let's develop a method to provide confidence intervals for our predictions:

In [ ]:
# Calculate prediction intervals based on historical error distribution
def calculate_prediction_interval(predicted_value, confidence=0.9):
    """
    Calculate prediction interval based on historical error distribution
    
    Parameters:
    -----------
    predicted_value : float
        The model's point prediction
    confidence : float, default=0.9
        Confidence level (e.g., 0.9 for 90% confidence interval)
        
    Returns:
    --------
    tuple
        (lower_bound, upper_bound)
    """
    # Calculate percentiles from the percentage error distribution
    lower_percentile = (1 - confidence) / 2
    upper_percentile = 1 - lower_percentile
    
    # Get the error percentiles
    lower_error_pct = np.percentile(percentage_errors, lower_percentile * 100)
    upper_error_pct = np.percentile(percentage_errors, upper_percentile * 100)
    
    # Calculate the bounds
    lower_bound = predicted_value * (1 + lower_error_pct / 100)
    upper_bound = predicted_value * (1 + upper_error_pct / 100)
    
    return lower_bound, upper_bound

In [ ]:
# Test the prediction interval function on a few examples
confidence_levels = [0.5, 0.8, 0.9, 0.95, 0.99]
test_prediction = 200000  # Example prediction

print(f"For a predicted value of ${test_prediction:,.2f}:")
for conf in confidence_levels:
    lower, upper = calculate_prediction_interval(test_prediction, conf)
    interval_width = upper - lower
    width_percentage = (interval_width / test_prediction) * 100
    print(f"{conf*100:.0f}% Confidence Interval: ${lower:,.2f} to ${upper:,.2f} (±{width_percentage:.1f}%)")

In [ ]:
# Visualize prediction intervals for a range of house prices
price_range = np.linspace(100000, 500000, 100)
confidence_levels = [0.5, 0.8, 0.9, 0.95]

plt.figure(figsize=(12, 8))

# Plot the prediction intervals for each confidence level
for conf in confidence_levels:
    lower_bounds = []
    upper_bounds = []
    
    for price in price_range:
        lower, upper = calculate_prediction_interval(price, conf)
        lower_bounds.append(lower)
        upper_bounds.append(upper)
    
    # Plot the confidence band
    plt.fill_between(price_range, lower_bounds, upper_bounds, alpha=0.2, label=f"{conf*100:.0f}% Confidence")

# Plot the perfect prediction line
plt.plot(price_range, price_range, 'k-', linewidth=2, label='Point Prediction')

plt.title('Prediction Intervals at Different Confidence Levels', fontsize=16)
plt.xlabel('Predicted Price ($)', fontsize=14)
plt.ylabel('Actual Price Range ($)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)

# Add annotations
plt.annotate('Wider intervals for\nhigher confidence levels', xy=(400000, 500000),
             xytext=(300000, 550000), arrowprops=dict(arrowstyle='->', lw=1.5),
             fontsize=12, bbox=dict(boxstyle='round,pad=0.5', fc='white', alpha=0.8))

plt.tight_layout()
plt.show()

## 3. Decision Support Systems for Real Estate Stakeholders

Let's design decision support systems for different real estate stakeholders based on our model.

### 3.1 Optimal Pricing Strategy for Sellers

Let's develop a pricing strategy tool for home sellers:

In [ ]:
def pricing_strategy_advisor(predicted_price, market_condition='balanced', risk_tolerance='moderate'):
    """
    Provide pricing strategy recommendations based on predicted price and market conditions
    
    Parameters:
    -----------
    predicted_price : float
        The model's predicted price
    market_condition : str, default='balanced'
        Market condition ('hot', 'balanced', or 'cold')
    risk_tolerance : str, default='moderate'
        Seller's risk tolerance ('conservative', 'moderate', or 'aggressive')
        
    Returns:
    --------
    dict
        Dictionary with pricing recommendations
    """
    # Calculate confidence intervals
    lower_90, upper_90 = calculate_prediction_interval(predicted_price, 0.9)
    lower_50, upper_50 = calculate_prediction_interval(predicted_price, 0.5)
    
    # Base adjustments on market conditions
    if market_condition == 'hot':
        base_adjustment = 0.05  # 5% premium in hot markets
    elif market_condition == 'cold':
        base_adjustment = -0.05  # 5% discount in cold markets
    else:  # balanced
        base_adjustment = 0.0
    
    # Adjust based on risk tolerance
    if risk_tolerance == 'conservative':
        risk_adjustment = -0.03  # 3% discount for quick sale
        target_percentile = lower_50
    elif risk_tolerance == 'aggressive':
        risk_adjustment = 0.03  # 3% premium to test market
        target_percentile = upper_50
    else:  # moderate
        risk_adjustment = 0.0
        target_percentile = predicted_price
    
    # Calculate recommended list price
    total_adjustment = base_adjustment + risk_adjustment
    recommended_price = target_percentile * (1 + total_adjustment)
    
    # Calculate expected days on market based on pricing strategy
    base_days = 30  # baseline days on market in balanced conditions
    
    if market_condition == 'hot':
        base_days *= 0.7  # 30% faster in hot markets
    elif market_condition == 'cold':
        base_days *= 1.5  # 50% slower in cold markets
    
    # Adjust days based on pricing strategy
    price_ratio = recommended_price / predicted_price
    if price_ratio > 1.05:  # Significantly overpriced
        days_multiplier = 1.5 + (price_ratio - 1.05) * 10  # Each 1% over adds 10% more time
    elif price_ratio < 0.95:  # Significantly underpriced
        days_multiplier = 0.7  # 30% faster for underpriced properties
    else:  # Reasonably priced
        days_multiplier = 1.0
    
    expected_days = base_days * days_multiplier
    
    # Calculate probability of selling within 30/60/90 days
    if expected_days <= 15:
        prob_30_days = 0.95
    elif expected_days <= 30:
        prob_30_days = 0.8
    elif expected_days <= 45:
        prob_30_days = 0.5
    else:
        prob_30_days = 0.3
    
    prob_60_days = min(prob_30_days + 0.3, 0.95)
    prob_90_days = min(prob_60_days + 0.2, 0.99)
    
    return {
        'predicted_price': predicted_price,
        'confidence_interval_90': (lower_90, upper_90),
        'recommended_list_price': recommended_price,
        'price_adjustment': total_adjustment * 100,  # Convert to percentage
        'expected_days_on_market': expected_days,
        'probability_sold_30_days': prob_30_days,
        'probability_sold_60_days': prob_60_days,
        'probability_sold_90_days': prob_90_days,
        'strategy_explanation': f"Based on a {'hot' if market_condition == 'hot' else 'cold' if market_condition == 'cold' else 'balanced'} market and your {risk_tolerance} risk tolerance, we recommend pricing at ${recommended_price:,.2f}."
    }

In [ ]:
# Test the pricing strategy advisor with different scenarios
test_price = 250000

# Create a table of different scenarios
scenarios = [
    {'market': 'hot', 'risk': 'conservative'},
    {'market': 'hot', 'risk': 'moderate'},
    {'market': 'hot', 'risk': 'aggressive'},
    {'market': 'balanced', 'risk': 'conservative'},
    {'market': 'balanced', 'risk': 'moderate'},
    {'market': 'balanced', 'risk': 'aggressive'},
    {'market': 'cold', 'risk': 'conservative'},
    {'market': 'cold', 'risk': 'moderate'},
    {'market': 'cold', 'risk': 'aggressive'}
]

# Generate recommendations for each scenario
recommendations = []
for scenario in scenarios:
    result = pricing_strategy_advisor(test_price, scenario['market'], scenario['risk'])
    recommendations.append({
        'Market': scenario['market'].capitalize(),
        'Risk Tolerance': scenario['risk'].capitalize(),
        'Recommended Price': f"${result['recommended_list_price']:,.0f}",
        'Price Adjustment': f"{result['price_adjustment']:+.1f}%",
        'Expected Days': f"{result['expected_days_on_market']:.0f}",
        'Prob. Sold in 30 Days': f"{result['probability_sold_30_days']:.0%}"
    })

# Convert to DataFrame and display
recommendations_df = pd.DataFrame(recommendations)
recommendations_df

In [ ]:
# Visualize the pricing recommendations
plt.figure(figsize=(12, 6))

# Extract data for plotting
market_conditions = ['Hot', 'Hot', 'Hot', 'Balanced', 'Balanced', 'Balanced', 'Cold', 'Cold', 'Cold']
risk_tolerances = ['Conservative', 'Moderate', 'Aggressive'] * 3
prices = [float(price.replace('$', '').replace(',', '')) for price in recommendations_df['Recommended Price']]

# Create a grouped bar chart
bar_width = 0.25
r1 = np.arange(3)
r2 = [x + bar_width for x in r1]
r3 = [x + bar_width for x in r2]

# Plot bars for each market condition
plt.bar(r1, prices[0:3], width=bar_width, label='Hot Market', color='salmon')
plt.bar(r2, prices[3:6], width=bar_width, label='Balanced Market', color='skyblue')
plt.bar(r3, prices[6:9], width=bar_width, label='Cold Market', color='lightgreen')

# Add labels and title
plt.xlabel('Risk Tolerance', fontsize=14)
plt.ylabel('Recommended List Price ($)', fontsize=14)
plt.title('Pricing Recommendations by Market Condition and Risk Tolerance', fontsize=16)
plt.xticks([r + bar_width for r in range(3)], ['Conservative', 'Moderate', 'Aggressive'])
plt.legend()

# Add price labels on top of bars
for i, price in enumerate(prices):
    x_pos = i % 3
    if i < 3:
        x_pos = r1[x_pos]
    elif i < 6:
        x_pos = r2[x_pos]
    else:
        x_pos = r3[x_pos]
    
    plt.text(x_pos, price + 5000, f'${price:,.0f}', ha='center', va='bottom', rotation=45, fontsize=10)

# Add reference line for the predicted price
plt.axhline(y=test_price, color='black', linestyle='--', label=f'Predicted Price: ${test_price:,.0f}')
plt.text(2.5, test_price + 5000, f'Predicted Price: ${test_price:,.0f}', ha='right', fontsize=12,
         bbox=dict(facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

### 3.2 Investment Analysis for Buyers

Let's create a tool to help buyers evaluate potential investments:

In [ ]:
def investment_analyzer(asking_price, predicted_price, annual_appreciation=0.03, 
                        holding_period=5, rental_yield=0.08, mortgage_rate=0.045,
                        down_payment_pct=0.2, closing_costs_pct=0.03, annual_expenses_pct=0.01):
    """
    Analyze a potential real estate investment
    
    Parameters:
    -----------
    asking_price : float
        The asking price of the property
    predicted_price : float
        The model's predicted fair market value
    annual_appreciation : float, default=0.03
        Expected annual appreciation rate
    holding_period : int, default=5
        Expected holding period in years
    rental_yield : float, default=0.08
        Annual rental income as percentage of property value
    mortgage_rate : float, default=0.045
        Annual mortgage interest rate
    down_payment_pct : float, default=0.2
        Down payment percentage
    closing_costs_pct : float, default=0.03
        Closing costs as percentage of purchase price
    annual_expenses_pct : float, default=0.01
        Annual maintenance and other expenses as percentage of property value
        
    Returns:
    --------
    dict
        Dictionary with investment analysis metrics
    """
    # Calculate initial metrics
    price_difference = predicted_price - asking_price
    price_difference_pct = price_difference / asking_price * 100
    
    # Calculate purchase costs
    down_payment = asking_price * down_payment_pct
    closing_costs = asking_price * closing_costs_pct
    initial_investment = down_payment + closing_costs
    loan_amount = asking_price - down_payment
    
    # Calculate monthly mortgage payment (P&I only)
    monthly_rate = mortgage_rate / 12
    num_payments = 30 * 12  # 30-year mortgage
    monthly_payment = loan_amount * (monthly_rate * (1 + monthly_rate) ** num_payments) / ((1 + monthly_rate) ** num_payments - 1)
    
    # Calculate annual cash flow
    annual_rental_income = asking_price * rental_yield
    annual_mortgage_payments = monthly_payment * 12
    annual_expenses = asking_price * annual_expenses_pct
    annual_cash_flow = annual_rental_income - annual_mortgage_payments - annual_expenses
    
    # Calculate cash-on-cash return
    cash_on_cash_return = annual_cash_flow / initial_investment * 100
    
    # Calculate future value after holding period
    future_value = asking_price * (1 + annual_appreciation) ** holding_period
    remaining_loan_balance = loan_amount * (1 - (1 - (1 + monthly_rate) ** (-num_payments)) / (1 - (1 + monthly_rate) ** (-num_payments + holding_period * 12)))
    
    # Calculate equity at sale
    equity_at_sale = future_value - remaining_loan_balance
    
    # Calculate selling costs
    selling_costs = future_value * 0.06  # Assume 6% selling costs
    
    # Calculate net profit
    net_profit_from_sale = equity_at_sale - down_payment - selling_costs
    total_cash_flow = annual_cash_flow * holding_period
    total_profit = net_profit_from_sale + total_cash_flow
    
    # Calculate ROI
    roi = total_profit / initial_investment * 100
    annualized_roi = ((1 + roi / 100) ** (1 / holding_period) - 1) * 100
    
    # Calculate cap rate
    net_operating_income = annual_rental_income - annual_expenses
    cap_rate = net_operating_income / asking_price * 100
    
    # Determine investment rating
    if price_difference_pct > 10 and cash_on_cash_return > 8 and annualized_roi > 12:
        rating = 'Excellent'
    elif price_difference_pct > 5 and cash_on_cash_return > 6 and annualized_roi > 10:
        rating = 'Good'
    elif price_difference_pct > 0 and cash_on_cash_return > 4 and annualized_roi > 8:
        rating = 'Fair'
    elif cash_on_cash_return > 0 and annualized_roi > 5:
        rating = 'Marginal'
    else:
        rating = 'Poor'
    
    return {
        'asking_price': asking_price,
        'predicted_price': predicted_price,
        'price_difference': price_difference,
        'price_difference_pct': price_difference_pct,
        'initial_investment': initial_investment,
        'monthly_payment': monthly_payment,
        'annual_cash_flow': annual_cash_flow,
        'cash_on_cash_return': cash_on_cash_return,
        'future_value': future_value,
        'equity_at_sale': equity_at_sale,
        'total_profit': total_profit,
        'roi': roi,
        'annualized_roi': annualized_roi,
        'cap_rate': cap_rate,
        'investment_rating': rating
    }

In [ ]:
# Test the investment analyzer with different scenarios
test_predicted_price = 250000

# Create a table of different asking prices
asking_prices = [220000, 235000, 250000, 265000, 280000]
analyses = []

for asking_price in asking_prices:
    result = investment_analyzer(asking_price, test_predicted_price)
    analyses.append({
        'Asking Price': f"${asking_price:,.0f}",
        'Predicted Value': f"${test_predicted_price:,.0f}",
        'Difference': f"${result['price_difference']:,.0f} ({result['price_difference_pct']:.1f}%)",
        'Monthly Payment': f"${result['monthly_payment']:,.0f}",
        'Annual Cash Flow': f"${result['annual_cash_flow']:,.0f}",
        'Cash-on-Cash Return': f"{result['cash_on_cash_return']:.1f}%",
        'Cap Rate': f"{result['cap_rate']:.1f}%",
        '5-Year ROI': f"{result['roi']:.1f}%",
        'Annualized ROI': f"{result['annualized_roi']:.1f}%",
        'Rating': result['investment_rating']
    })

# Convert to DataFrame and display
analyses_df = pd.DataFrame(analyses)
analyses_df

In [ ]:
# Visualize the investment metrics across different asking prices
plt.figure(figsize=(14, 8))

# Extract data for plotting
asking_prices_raw = [float(price.replace('$', '').replace(',', '')) for price in analyses_df['Asking Price']]
cash_on_cash = [float(rate.replace('%', '')) for rate in analyses_df['Cash-on-Cash Return']]
cap_rates = [float(rate.replace('%', '')) for rate in analyses_df['Cap Rate']]
annualized_roi = [float(rate.replace('%', '')) for rate in analyses_df['Annualized ROI']]

# Create subplots
plt.subplot(2, 1, 1)
plt.plot(asking_prices_raw, cash_on_cash, 'b-o', linewidth=2, label='Cash-on-Cash Return')
plt.plot(asking_prices_raw, cap_rates, 'g-s', linewidth=2, label='Cap Rate')
plt.plot(asking_prices_raw, annualized_roi, 'r-^', linewidth=2, label='Annualized ROI')
plt.axvline(x=test_predicted_price, color='black', linestyle='--', label='Predicted Fair Value')
plt.title('Investment Metrics by Asking Price', fontsize=16)
plt.ylabel('Return Rate (%)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)

# Add value labels
for i, (price, coc, cap, roi) in enumerate(zip(asking_prices_raw, cash_on_cash, cap_rates, annualized_roi)):
    plt.text(price, coc + 0.2, f"{coc:.1f}%", ha='center', fontsize=10)
    plt.text(price, cap + 0.2, f"{cap:.1f}%", ha='center', fontsize=10)
    plt.text(price, roi + 0.2, f"{roi:.1f}%", ha='center', fontsize=10)

# Extract price difference percentages
price_diff_pct = []
for diff in analyses_df['Difference']:
    # Extract the percentage from the string
    pct = float(diff.split('(')[1].split('%')[0])
    price_diff_pct.append(pct)

# Create a bar chart of price differences
plt.subplot(2, 1, 2)
bars = plt.bar(asking_prices_raw, price_diff_pct, color=['green' if pct > 0 else 'red' for pct in price_diff_pct])
plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
plt.title('Price Difference (Predicted vs. Asking)', fontsize=16)
plt.xlabel('Asking Price ($)', fontsize=14)
plt.ylabel('Price Difference (%)', fontsize=14)
plt.grid(True, alpha=0.3)

# Add value labels
for bar, pct in zip(bars, price_diff_pct):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.5 if height > 0 else height - 1.5,
             f"{pct:.1f}%", ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 4. Case Studies: Applying the Model to Specific Business Scenarios

Let's explore how our model can be applied to specific real-world business scenarios.

### 4.1 Case Study 1: Real Estate Agency Optimization

A real estate agency wants to use our model to optimize their listing and marketing strategies.

In [ ]:
# Simulate a portfolio of properties for a real estate agency
np.random.seed(42)  # For reproducibility

# Generate 20 properties with different characteristics
properties = []
for i in range(20):
    # Generate random property characteristics
    overall_qual = np.random.choice([5, 6, 7, 8, 9])
    gr_liv_area = np.random.randint(1200, 3000)
    year_built = np.random.randint(1950, 2010)
    neighborhood = np.random.choice(['NAmes', 'Edwards', 'BrkSide', 'NoRidge', 'StoneBr'])
    
    # Generate a "true" value based on these characteristics
    base_price = 150000
    quality_factor = (overall_qual - 5) * 20000
    size_factor = (gr_liv_area - 1500) * 50
    age_factor = (year_built - 1980) * 500
    neighborhood_factor = {
        'NAmes': 0,
        'Edwards': -20000,
        'BrkSide': -10000,
        'NoRidge': 50000,
        'StoneBr': 40000
    }[neighborhood]
    
    true_value = base_price + quality_factor + size_factor + age_factor + neighborhood_factor
    true_value = max(true_value, 100000)  # Ensure minimum value
    
    # Add some random noise to simulate market variability
    true_value *= np.random.uniform(0.95, 1.05)
    
    # Generate an asking price with some bias
    asking_price = true_value * np.random.uniform(0.9, 1.15)
    
    # Generate a model prediction with some error
    predicted_value = true_value * np.random.uniform(0.9, 1.1)
    
    # Calculate days on market based on price difference
    price_diff_pct = (asking_price - true_value) / true_value
    base_days = 30
    if price_diff_pct > 0.1:  # Significantly overpriced
        days_on_market = base_days * (1 + price_diff_pct * 10)
    elif price_diff_pct < -0.05:  # Significantly underpriced
        days_on_market = base_days * 0.7
    else:  # Reasonably priced
        days_on_market = base_days * (1 + price_diff_pct * 5)
    
    days_on_market = int(days_on_market)
    
    # Add property to the list
    properties.append({
        'property_id': f"P{i+1:02d}",
        'neighborhood': neighborhood,
        'overall_qual': overall_qual,
        'gr_liv_area': gr_liv_area,
        'year_built': year_built,
        'true_value': true_value,
        'asking_price': asking_price,
        'predicted_value': predicted_value,
        'days_on_market': days_on_market,
        'status': 'Active' if np.random.random() > 0.3 else 'Pending'
    })

# Convert to DataFrame
properties_df = pd.DataFrame(properties)

# Calculate additional metrics
properties_df['price_diff'] = properties_df['predicted_value'] - properties_df['asking_price']
properties_df['price_diff_pct'] = properties_df['price_diff'] / properties_df['asking_price'] * 100

# Display the properties
display_df = properties_df[['property_id', 'neighborhood', 'overall_qual', 'gr_liv_area', 
                           'asking_price', 'predicted_value', 'price_diff_pct', 'days_on_market', 'status']].copy()

# Format the numeric columns
display_df['asking_price'] = display_df['asking_price'].apply(lambda x: f"${x:,.0f}")
display_df['predicted_value'] = display_df['predicted_value'].apply(lambda x: f"${x:,.0f}")
display_df['price_diff_pct'] = display_df['price_diff_pct'].apply(lambda x: f"{x:.1f}%")

display_df

In [ ]:
# Analyze the portfolio and provide recommendations
def analyze_portfolio(properties_df):
    """
    Analyze a portfolio of properties and provide recommendations
    
    Parameters:
    -----------
    properties_df : DataFrame
        DataFrame containing property information
        
    Returns:
    --------
    dict
        Dictionary with portfolio analysis and recommendations
    """
    # Calculate portfolio metrics
    total_asking = properties_df['asking_price'].sum()
    total_predicted = properties_df['predicted_value'].sum()
    avg_days_on_market = properties_df['days_on_market'].mean()
    
    # Identify overpriced properties (predicted < asking by more than 5%)
    overpriced = properties_df[properties_df['price_diff_pct'] < -5].copy()
    overpriced['recommended_price'] = overpriced['predicted_value'] * 1.02  # Slight premium
    overpriced['price_reduction'] = overpriced['asking_price'] - overpriced['recommended_price']
    overpriced['price_reduction_pct'] = overpriced['price_reduction'] / overpriced['asking_price'] * 100
    
    # Identify underpriced properties (predicted > asking by more than 5%)
    underpriced = properties_df[properties_df['price_diff_pct'] > 5].copy()
    
    # Identify properties with excessive days on market
    long_dom = properties_df[properties_df['days_on_market'] > 45].copy()
    
    # Calculate potential commission impact
    commission_rate = 0.06  # 6% typical commission
    current_commission = total_asking * commission_rate
    
    # Estimate time to sell for overpriced properties
    avg_time_reduction = 15  # days saved per property with price adjustment
    total_time_saved = len(overpriced) * avg_time_reduction
    
    # Estimate impact of price adjustments
    if not overpriced.empty:
        adjusted_asking = total_asking - overpriced['price_reduction'].sum()
        adjusted_commission = adjusted_asking * commission_rate
        commission_impact = adjusted_commission - current_commission
    else:
        adjusted_asking = total_asking
        adjusted_commission = current_commission
        commission_impact = 0
    
    # Calculate potential revenue from faster sales
    avg_listings_per_month = 10
    additional_listings = total_time_saved / 30 * avg_listings_per_month
    additional_revenue = additional_listings * (adjusted_asking / len(properties_df)) * commission_rate
    
    return {
        'total_properties': len(properties_df),
        'active_properties': len(properties_df[properties_df['status'] == 'Active']),
        'pending_properties': len(properties_df[properties_df['status'] == 'Pending']),
        'total_asking': total_asking,
        'total_predicted': total_predicted,
        'avg_days_on_market': avg_days_on_market,
        'overpriced_properties': len(overpriced),
        'underpriced_properties': len(underpriced),
        'long_dom_properties': len(long_dom),
        'price_reduction_recommendations': overpriced,
        'current_commission': current_commission,
        'adjusted_commission': adjusted_commission,
        'commission_impact': commission_impact,
        'total_time_saved': total_time_saved,
        'additional_revenue': additional_revenue,
        'net_benefit': additional_revenue + commission_impact
    }

In [ ]:
# Analyze the portfolio
portfolio_analysis = analyze_portfolio(properties_df)

# Display the analysis summary
print("Portfolio Analysis Summary:")
print(f"Total Properties: {portfolio_analysis['total_properties']}")
print(f"Active Listings: {portfolio_analysis['active_properties']}")
print(f"Pending Sales: {portfolio_analysis['pending_properties']}")
print(f"Total Asking Price: ${portfolio_analysis['total_asking']:,.2f}")
print(f"Total Predicted Value: ${portfolio_analysis['total_predicted']:,.2f}")
print(f"Average Days on Market: {portfolio_analysis['avg_days_on_market']:.1f}")
print(f"\nProperties Requiring Attention:")
print(f"Overpriced Properties: {portfolio_analysis['overpriced_properties']}")
print(f"Underpriced Properties: {portfolio_analysis['underpriced_properties']}")
print(f"Properties with Excessive Days on Market: {portfolio_analysis['long_dom_properties']}")

# Display financial impact
print(f"\nFinancial Impact of Recommendations:")
print(f"Current Expected Commission: ${portfolio_analysis['current_commission']:,.2f}")
print(f"Commission Impact of Price Adjustments: ${portfolio_analysis['commission_impact']:,.2f}")
print(f"Estimated Time Saved: {portfolio_analysis['total_time_saved']} days")
print(f"Additional Revenue from Faster Sales: ${portfolio_analysis['additional_revenue']:,.2f}")
print(f"Net Benefit: ${portfolio_analysis['net_benefit']:,.2f}")

In [ ]:
# Display price reduction recommendations
if portfolio_analysis['overpriced_properties'] > 0:
    price_reductions = portfolio_analysis['price_reduction_recommendations'][
        ['property_id', 'neighborhood', 'asking_price', 'predicted_value', 
         'recommended_price', 'price_reduction', 'price_reduction_pct', 'days_on_market']
    ].copy()
    
    # Format the numeric columns
    price_reductions['asking_price'] = price_reductions['asking_price'].apply(lambda x: f"${x:,.0f}")
    price_reductions['predicted_value'] = price_reductions['predicted_value'].apply(lambda x: f"${x:,.0f}")
    price_reductions['recommended_price'] = price_reductions['recommended_price'].apply(lambda x: f"${x:,.0f}")
    price_reductions['price_reduction'] = price_reductions['price_reduction'].apply(lambda x: f"${x:,.0f}")
    price_reductions['price_reduction_pct'] = price_reductions['price_reduction_pct'].apply(lambda x: f"{x:.1f}%")
    
    print("\nRecommended Price Reductions:")
    display(price_reductions)
else:
    print("\nNo price reduction recommendations at this time.")

### 4.2 Case Study 2: Home Renovation ROI Analysis

A homeowner wants to know which renovations will provide the best return on investment when selling their home.

In [ ]:
# Define a function to analyze potential renovations
def analyze_renovation_roi(base_features, renovation_options):
    """
    Analyze the ROI of different renovation options
    
    Parameters:
    -----------
    base_features : dict
        Dictionary of current home features
    renovation_options : list
        List of dictionaries containing renovation options
        
    Returns:
    --------
    DataFrame
        DataFrame with renovation ROI analysis
    """
    # Create a copy of base features for each renovation option
    results = []
    
    # Get baseline predicted value
    baseline_value = base_features['predicted_value']
    
    for option in renovation_options:
        # Apply the renovation changes to the base features
        modified_features = base_features.copy()
        
        # Update the features based on the renovation
        for feature, change in option['feature_changes'].items():
            if feature in modified_features:
                modified_features[feature] += change
            else:
                modified_features[feature] = change
        
        # Simulate a new prediction
        # In a real implementation, this would call the actual model
        # Here we'll use a simple heuristic based on the renovation impact
        new_value = baseline_value * (1 + option['expected_value_increase'])
        
        # Calculate ROI
        value_increase = new_value - baseline_value
        roi = (value_increase / option['cost'] - 1) * 100
        
        # Add to results
        results.append({
            'renovation': option['name'],
            'cost': option['cost'],
            'baseline_value': baseline_value,
            'new_value': new_value,
            'value_increase': value_increase,
            'roi': roi,
            'payback_ratio': value_increase / option['cost'],
            'description': option['description']
        })
    
    # Convert to DataFrame and sort by ROI
    results_df = pd.DataFrame(results).sort_values('roi', ascending=False)
    return results_df

In [ ]:
# Define a sample home with current features
sample_home = {
    'gr_liv_area': 1800,
    'overall_qual': 6,
    'year_built': 1985,
    'full_bath': 2,
    'kitchen_qual': 'TA',  # Typical/Average
    'exter_qual': 'TA',    # Typical/Average
    'total_bsmt_sf': 900,
    'garage_cars': 2,
    'predicted_value': 275000
}

# Define potential renovation options
renovation_options = [
    {
        'name': 'Kitchen Remodel',
        'cost': 30000,
        'expected_value_increase': 0.07,  # 7% increase in home value
        'feature_changes': {
            'kitchen_qual': 1  # Upgrade from TA to Gd
        },
        'description': 'Complete kitchen renovation with new cabinets, countertops, and appliances'
    },
    {
        'name': 'Bathroom Remodel',
        'cost': 15000,
        'expected_value_increase': 0.04,  # 4% increase in home value
        'feature_changes': {
            'full_bath': 0  # No change in number of bathrooms
        },
        'description': 'Update existing bathrooms with new fixtures, tile, and vanities'
    },
    {
        'name': 'Add Bathroom',
        'cost': 25000,
        'expected_value_increase': 0.05,  # 5% increase in home value
        'feature_changes': {
            'full_bath': 1  # Add one bathroom
        },
        'description': 'Add a new full bathroom'
    },
    {
        'name': 'Finish Basement',
        'cost': 35000,
        'expected_value_increase': 0.075,  # 7.5% increase in home value
        'feature_changes': {
            'gr_liv_area': 900  # Add basement area to living area
        },
        'description': 'Finish the basement to add living space'
    },
    {
        'name': 'Exterior Improvements',
        'cost': 20000,
        'expected_value_increase': 0.05,  # 5% increase in home value
        'feature_changes': {
            'exter_qual': 1  # Upgrade from TA to Gd
        },
        'description': 'New siding, windows, and front door'
    },
    {
        'name': 'Add Garage Space',
        'cost': 28000,
        'expected_value_increase': 0.04,  # 4% increase in home value
        'feature_changes': {
            'garage_cars': 1  # Add space for one more car
        },
        'description': 'Expand garage to fit an additional car'
    },
    {
        'name': 'Overall Quality Improvements',
        'cost': 40000,
        'expected_value_increase': 0.09,  # 9% increase in home value
        'feature_changes': {
            'overall_qual': 1  # Increase overall quality by 1 point
        },
        'description': 'General improvements throughout the home to increase overall quality'
    },
    {
        'name': 'Room Addition',
        'cost': 50000,
        'expected_value_increase': 0.08,  # 8% increase in home value
        'feature_changes': {
            'gr_liv_area': 400  # Add 400 sq ft
        },
        'description': 'Add a new room or expand existing living space'
    }
]

# Analyze renovation options
renovation_analysis = analyze_renovation_roi(sample_home, renovation_options)

# Format the results for display
display_renovation = renovation_analysis[['renovation', 'cost', 'value_increase', 'roi', 'payback_ratio', 'description']].copy()
display_renovation['cost'] = display_renovation['cost'].apply(lambda x: f"${x:,.0f}")
display_renovation['value_increase'] = display_renovation['value_increase'].apply(lambda x: f"${x:,.0f}")
display_renovation['roi'] = display_renovation['roi'].apply(lambda x: f"{x:.1f}%")
display_renovation['payback_ratio'] = display_renovation['payback_ratio'].apply(lambda x: f"{x:.2f}")

# Display the results
print(f"Renovation ROI Analysis for Home Valued at ${sample_home['predicted_value']:,.0f}")
display_renovation

In [ ]:
# Visualize the renovation ROI analysis
plt.figure(figsize=(12, 8))

# Extract data for plotting
renovations = renovation_analysis['renovation'].tolist()
costs = renovation_analysis['cost'].values
value_increases = renovation_analysis['value_increase'].values
rois = renovation_analysis['roi'].values

# Sort by ROI
sorted_indices = np.argsort(rois)[::-1]
renovations = [renovations[i] for i in sorted_indices]
costs = [costs[i] for i in sorted_indices]
value_increases = [value_increases[i] for i in sorted_indices]
rois = [rois[i] for i in sorted_indices]

# Create a bar chart of costs and value increases
x = np.arange(len(renovations))
width = 0.35

fig, ax1 = plt.subplots(figsize=(14, 8))

# Plot bars for cost and value increase
bars1 = ax1.bar(x - width/2, costs, width, label='Renovation Cost', color='salmon')
bars2 = ax1.bar(x + width/2, value_increases, width, label='Value Increase', color='skyblue')

# Add a second y-axis for ROI
ax2 = ax1.twinx()
ax2.plot(x, rois, 'ko-', linewidth=2, label='ROI')

# Add labels and title
ax1.set_xlabel('Renovation Option', fontsize=14)
ax1.set_ylabel('Amount ($)', fontsize=14)
ax2.set_ylabel('Return on Investment (%)', fontsize=14)
ax1.set_title('Renovation Costs, Value Increases, and ROI', fontsize=16)
ax1.set_xticks(x)
ax1.set_xticklabels(renovations, rotation=45, ha='right')

# Add a horizontal line at ROI = 0%
ax2.axhline(y=0, color='red', linestyle='--', alpha=0.5)

# Add value labels
for i, (cost, value, roi) in enumerate(zip(costs, value_increases, rois)):
    ax1.text(i - width/2, cost + 1000, f'${cost:,.0f}', ha='center', va='bottom', rotation=45, fontsize=10)
    ax1.text(i + width/2, value + 1000, f'${value:,.0f}', ha='center', va='bottom', rotation=45, fontsize=10)
    ax2.text(i, roi + 2, f'{roi:.1f}%', ha='center', va='bottom', fontsize=10)

# Add legends
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

## Summary

In this notebook, we've explored how to translate our machine learning model into real business value through:

1. **Cost-Benefit Analysis**: We quantified the costs and benefits of implementing our house price prediction model, calculating ROI and payback period.

2. **Risk Assessment**: We identified potential risks, created a risk matrix, and developed methods to quantify prediction uncertainty through confidence intervals.

3. **Decision Support Systems**: We built tools to help real estate stakeholders make better decisions, including a pricing strategy advisor for sellers and an investment analyzer for buyers.

4. **Case Studies**: We applied our model to specific business scenarios, demonstrating how it can optimize a real estate agency's portfolio and help homeowners prioritize renovation projects.

These applications demonstrate that the value of machine learning models extends far beyond technical metrics like R² or RMSE. By connecting our model to business outcomes and decision-making processes, we can create significant value for various stakeholders in the real estate market.

The key takeaway is that successful machine learning projects require both technical excellence and business acumen. Understanding the business context, stakeholder needs, and potential risks is just as important as building an accurate model.